# 01 — Data Collection: SpaceX API

IBM Data Science Capstone — SpaceX Falcon 9 first-stage landing prediction.

**Objective.** Collect historical SpaceX launch data from the public SpaceX REST API, normalize the nested launch/core/payload/launchpad fields, and isolate Falcon 9 launches through the capstone study period.

The repository also includes the 90-row capstone API extract used for the reproducible analysis in this submission.

In [ ]:
import requests, pandas as pd
from pathlib import Path

API = 'https://api.spacexdata.com/v4'
launches = requests.get(f'{API}/launches/past', timeout=30).json()
len(launches)

In [ ]:
# Identify Falcon 9 rocket IDs, then query launches with populated rocket/core/payload/launchpad documents.
rockets = requests.get(f'{API}/rockets', timeout=30).json()
f9_ids = {r['id'] for r in rockets if r.get('name') == 'Falcon 9'}
query = {
    'query': {'rocket': {'$in': list(f9_ids)}},
    'options': {'limit': 1000, 'sort': {'flight_number': 'asc'},
                'populate': ['rocket','payloads','launchpad','cores']}
}
r = requests.post(f'{API}/launches/query', json=query, timeout=30)
r.raise_for_status()
data = r.json()['docs']
len(data)

In [ ]:
# Save a compact normalized extract. The exact historical course extract is kept separately in data/dataset_part_1.csv.
rows=[]
for x in data:
    core=(x.get('cores') or [{}])[0] or {}
    pad=x.get('launchpad') or {}
    payload=(x.get('payloads') or [{}])[0] or {}
    rows.append({
        'FlightNumber': x.get('flight_number'),
        'Date': x.get('date_utc'),
        'BoosterVersion': (x.get('rocket') or {}).get('name'),
        'PayloadMass': payload.get('mass_kg'),
        'Orbit': payload.get('orbit'),
        'LaunchSite': pad.get('name'),
        'Outcome': f"{core.get('landing_success')} {core.get('landing_type')}",
        'Flights': core.get('flight'),
        'GridFins': core.get('gridfins'),
        'Reused': core.get('reused'),
        'Legs': core.get('legs'),
        'LandingPad': core.get('landpad'),
        'Serial': core.get('core'),
    })
api_df=pd.DataFrame(rows)
api_df.head()

**Reproducible course extract:** `data/dataset_part_1.csv` contains 90 Falcon 9 rows from the standard IBM capstone dataset used for the remaining analysis. The public API documentation specifies the `/v4/launches` and `/v4/launches/query` endpoints. See the project README for source links.